# Federated Learning: Worker Selection Strategies for Communication Efficiency
## Reducing Communication by Selecting Fewer Workers

**Research Question:** How do different numbers of workers and selection strategies affect FL efficiency?

**Independent Variables:**
1. **Number of Workers (Total Pool):** {10, 20, 50, 100}
2. **Worker Selection Strategies:**
   - Random Selection
   - Cyclic Selection (round-robin)
   - Adaptive Selection (LAG - Least Recently Selected)

**Dependent Variables:**
- Communication cost (bits)
- Model accuracy (90% threshold)
- Convergence round
- Training time

---

## 1. Setup & Installation

In [1]:
# Check GPU
!nvidia-smi

# Install packages
!pip install torch torchvision matplotlib seaborn -q

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
import torchvision
import torchvision.transforms as transforms
import numpy as np
import time
import copy
import json
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple
from collections import deque

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# Configure plotting
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)

# Check device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"\n{'='*60}")
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"{'='*60}\n")

Tue Dec  9 22:15:08 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Worker Selection Strategies

In [2]:
class WorkerSelectionStrategy:
    """Base class for worker selection strategies"""
    def __init__(self, name: str):
        self.name = name

    def select_workers(self, workers: List, num_select: int, round_idx: int) -> List:
        """Select workers for the current round"""
        raise NotImplementedError


class RandomSelection(WorkerSelectionStrategy):
    """Randomly select workers each round"""
    def __init__(self):
        super().__init__("Random")

    def select_workers(self, workers: List, num_select: int, round_idx: int) -> List:
        return list(np.random.choice(workers, num_select, replace=False))


class CyclicSelection(WorkerSelectionStrategy):
    """Select workers in round-robin fashion"""
    def __init__(self):
        super().__init__("Cyclic")
        self.current_idx = 0

    def select_workers(self, workers: List, num_select: int, round_idx: int) -> List:
        num_workers = len(workers)
        selected = []

        for _ in range(num_select):
            selected.append(workers[self.current_idx % num_workers])
            self.current_idx += 1

        return selected


class AdaptiveLAGSelection(WorkerSelectionStrategy):
    """Adaptive selection: prioritize Least Recently Selected workers (LAG)"""
    def __init__(self):
        super().__init__("Adaptive-LAG")
        self.last_selected_round = {}  # Track when each worker was last selected

    def select_workers(self, workers: List, num_select: int, round_idx: int) -> List:
        # Initialize tracking for new workers
        for worker in workers:
            if worker not in self.last_selected_round:
                self.last_selected_round[worker] = -1  # Never selected

        # Calculate "lag" (rounds since last selection)
        worker_lags = [(worker, round_idx - self.last_selected_round[worker])
                       for worker in workers]

        # Sort by lag (descending) - prioritize workers not selected recently
        worker_lags.sort(key=lambda x: x[1], reverse=True)

        # Select top num_select workers with highest lag
        selected = [worker for worker, _ in worker_lags[:num_select]]

        # Update last selected round
        for worker in selected:
            self.last_selected_round[worker] = round_idx

        return selected


print("✓ Selection strategies defined!")

✓ Selection strategies defined!


## 3. Model & FL Components

In [3]:
class SimpleCNN(nn.Module):
    """Simple CNN for MNIST"""
    def __init__(self, num_classes=10, input_channels=1):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(input_channels, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, num_classes)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


class FederatedClient:
    """Federated Learning Client"""
    def __init__(self, client_id: int, train_data: Dataset, device: str):
        self.client_id = client_id
        self.train_data = train_data
        self.device = device
        self.model = None

    def set_model(self, model: nn.Module):
        self.model = copy.deepcopy(model).to(self.device)

    def train(self, epochs: int, batch_size: int, learning_rate: float) -> Tuple[Dict, float]:
        self.model.train()
        optimizer = optim.SGD(self.model.parameters(), lr=learning_rate, momentum=0.9)
        criterion = nn.CrossEntropyLoss()

        train_loader = DataLoader(self.train_data, batch_size=batch_size, shuffle=True)

        total_computation_time = 0

        for epoch in range(epochs):
            epoch_start = time.time()
            for data, target in train_loader:
                data, target = data.to(self.device), target.to(self.device)
                optimizer.zero_grad()
                output = self.model(data)
                loss = criterion(output, target)
                loss.backward()
                optimizer.step()
            total_computation_time += time.time() - epoch_start

        model_params = {name: param.cpu().clone() for name, param in self.model.named_parameters()}

        return model_params, total_computation_time


class FederatedServer:
    """Federated Learning Server"""
    def __init__(self, model: nn.Module, test_data: Dataset, device: str):
        self.global_model = model.to(device)
        self.test_data = test_data
        self.device = device

    def aggregate(self, client_params_list: List[Dict], client_weights: List[float]):
        global_params = self.global_model.state_dict()

        for key in global_params.keys():
            if 'weight' in key or 'bias' in key:
                global_params[key] = torch.zeros_like(global_params[key])
                for client_params, weight in zip(client_params_list, client_weights):
                    global_params[key] += client_params[key].to(self.device) * weight

        self.global_model.load_state_dict(global_params)

    def evaluate(self) -> Tuple[float, float]:
        self.global_model.eval()
        test_loader = DataLoader(self.test_data, batch_size=128, shuffle=False)

        correct = 0
        total = 0
        total_loss = 0
        criterion = nn.CrossEntropyLoss()

        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(self.device), target.to(self.device)
                outputs = self.global_model(data)
                loss = criterion(outputs, target)
                total_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total += target.size(0)
                correct += (predicted == target).sum().item()

        accuracy = 100 * correct / total
        avg_loss = total_loss / len(test_loader)
        return accuracy, avg_loss

    def get_model_size_bits(self) -> int:
        total_bits = 0
        for param in self.global_model.parameters():
            total_bits += param.numel() * 32
        return total_bits


def create_non_iid_split(dataset, num_clients: int, alpha: float = 0.5):
    """Create non-IID data split"""
    labels = np.array([label for _, label in dataset])
    num_classes = len(np.unique(labels))

    client_indices = [[] for _ in range(num_clients)]

    for k in range(num_classes):
        idx_k = np.where(labels == k)[0]
        np.random.shuffle(idx_k)

        proportions = np.random.dirichlet(np.repeat(alpha, num_clients))
        proportions = (np.cumsum(proportions) * len(idx_k)).astype(int)[:-1]

        idx_k_split = np.split(idx_k, proportions)
        for i, idx in enumerate(idx_k_split):
            client_indices[i].extend(idx)

    client_datasets = [Subset(dataset, indices) for indices in client_indices]

    return client_datasets

print("✓ Model and FL classes defined!")

✓ Model and FL classes defined!


## 4. Main Experiment Function

In [4]:
def run_worker_selection_experiment(
    num_total_workers: int,
    selection_strategy: WorkerSelectionStrategy,
    workers_per_round: int = None,  # If None, use 30% of total
    num_rounds: int = 50,
    local_epochs: int = 5,
    batch_size: int = 32,
    learning_rate: float = 0.01,
    target_accuracy: float = 90.0,
    device: str = 'cuda'
) -> Dict:
    """
    Run FL experiment with specified number of workers and selection strategy
    """
    # Calculate workers per round if not specified
    if workers_per_round is None:
        workers_per_round = max(1, int(0.3 * num_total_workers))

    print(f"\n{'='*60}")
    print(f"Workers: {num_total_workers} | Selection: {selection_strategy.name}")
    print(f"Selecting {workers_per_round} workers per round")
    print(f"{'='*60}")

    # Load MNIST dataset
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])

    train_dataset = torchvision.datasets.MNIST(
        root='./data', train=True, download=True, transform=transform
    )
    train_subset_indices = np.random.choice(len(train_dataset), len(train_dataset)//10, replace=False)
    train_dataset = Subset(train_dataset, train_subset_indices)
    test_dataset = torchvision.datasets.MNIST(
        root='./data', train=False, download=True, transform=transform
    )

    # Create non-IID data split
    print("Creating non-IID data split...")
    client_datasets = create_non_iid_split(train_dataset, num_total_workers, alpha=0.5)

    # Initialize model
    model = SimpleCNN(num_classes=10, input_channels=1)

    # Initialize server and clients
    server = FederatedServer(model, test_dataset, device)
    clients = [FederatedClient(i, client_datasets[i], device) for i in range(num_total_workers)]

    # Metrics tracking
    results = {
        'num_workers': num_total_workers,
        'selection_strategy': selection_strategy.name,
        'workers_per_round': workers_per_round,
        'accuracy_history': [],
        'loss_history': [],
        'training_time_history': [],
        'communication_cost_history': [],
        'computation_cost_history': [],
        'round_history': [],
        'convergence_round': None,
        'total_communication_cost': 0,
        'total_computation_cost': 0,
        'total_training_time': 0,
        'worker_participation': {i: 0 for i in range(num_total_workers)}  # Track how often each worker participates
    }

    model_size_bits = server.get_model_size_bits()

    cumulative_time = 0
    cumulative_comm_cost = 0
    cumulative_comp_cost = 0

    # Training loop
    for round_idx in range(num_rounds):
        round_start = time.time()

        # Select workers using strategy
        selected_clients = selection_strategy.select_workers(clients, workers_per_round, round_idx)

        # Track participation
        for client in selected_clients:
            results['worker_participation'][client.client_id] += 1

        # Client training
        client_params_list = []
        client_weights = []
        round_comp_time = 0

        for client in selected_clients:
            client.set_model(server.global_model)

            params, comp_time = client.train(
                epochs=local_epochs,
                batch_size=batch_size,
                learning_rate=learning_rate
            )

            client_params_list.append(params)
            weight = len(client.train_data) / sum(len(c.train_data) for c in selected_clients)
            client_weights.append(weight)
            round_comp_time += comp_time

        # Server aggregation
        server.aggregate(client_params_list, client_weights)

        # Communication cost: upload + download for selected workers
        round_comm_cost = model_size_bits * workers_per_round * 2

        # Evaluation
        accuracy, loss = server.evaluate()

        round_time = time.time() - round_start

        # Update cumulative metrics
        cumulative_time += round_time
        cumulative_comm_cost += round_comm_cost
        cumulative_comp_cost += round_comp_time

        # Store metrics
        results['accuracy_history'].append(accuracy)
        results['loss_history'].append(loss)
        results['training_time_history'].append(cumulative_time)
        results['communication_cost_history'].append(cumulative_comm_cost)
        results['computation_cost_history'].append(cumulative_comp_cost)
        results['round_history'].append(round_idx + 1)

        # Check convergence
        if accuracy >= target_accuracy and results['convergence_round'] is None:
            results['convergence_round'] = round_idx + 1
            print(f"✓ Converged at round {round_idx + 1} with accuracy {accuracy:.2f}%")

        if (round_idx + 1) % 5 == 0 or round_idx == 0:
            print(f"Round {round_idx + 1:3d} | Acc: {accuracy:6.2f}% | Loss: {loss:.4f} | "
                  f"Time: {cumulative_time:6.1f}s | Comm: {cumulative_comm_cost/1e9:.3f}Gb")

        # Early stopping
        if results['convergence_round'] is not None and round_idx >= results['convergence_round'] + 10:
            print(f"Stopping early after convergence...")
            break

    results['total_training_time'] = cumulative_time
    results['total_communication_cost'] = cumulative_comm_cost
    results['total_computation_cost'] = cumulative_comp_cost

    # Calculate participation statistics
    participations = list(results['worker_participation'].values())
    results['participation_stats'] = {
        'mean': np.mean(participations),
        'std': np.std(participations),
        'min': np.min(participations),
        'max': np.max(participations)
    }

    print(f"\nFinal Results:")
    print(f"  Final Accuracy: {results['accuracy_history'][-1]:.2f}%")
    print(f"  Convergence Round: {results['convergence_round']}")
    print(f"  Total Training Time: {results['total_training_time']:.2f}s")
    print(f"  Total Communication Cost: {results['total_communication_cost']/1e9:.3f} Gb")
    print(f"  Worker Participation (mean±std): {results['participation_stats']['mean']:.1f}±{results['participation_stats']['std']:.1f}")

    return results

print("✓ Experiment function defined!")

✓ Experiment function defined!


## 5. Experiment 1: Effect of Number of Workers

Test with {5, 10, 25} workers using Random selection

In [7]:
# Test different numbers of workers (with Random selection)
num_workers_list = [10, 25, 50]
worker_results = {}

print("\n" + "="*60)
print("EXPERIMENT 1: NUMBER OF WORKERS (Random Selection)")
print("="*60)

for num_workers in num_workers_list:
    results = run_worker_selection_experiment(
        num_total_workers=num_workers,
        selection_strategy=RandomSelection(),
        workers_per_round=None,  # 30% of total
        num_rounds=100,
        local_epochs=5,
        batch_size=512,
        learning_rate=0.01,
        target_accuracy=95.0,
        device=device
    )
    worker_results[f'{num_workers}_workers_Random'] = results

print("\n" + "="*60)
print("✓ Worker count experiments completed!")
print("="*60)


EXPERIMENT 1: NUMBER OF WORKERS (Random Selection)

Workers: 5 | Selection: Random
Selecting 1 workers per round
Creating non-IID data split...
Round   1 | Acc:  15.22% | Loss: 6.4574 | Time:    3.5s | Comm: 0.027Gb
Round   5 | Acc:  10.32% | Loss: 2.3035 | Time:   16.1s | Comm: 0.135Gb
Round  10 | Acc:   9.74% | Loss: 2.2991 | Time:   31.0s | Comm: 0.270Gb
Round  15 | Acc:  10.28% | Loss: 2.3109 | Time:   46.7s | Comm: 0.405Gb
Round  20 | Acc:  12.60% | Loss: 3.2371 | Time:   63.7s | Comm: 0.540Gb
Round  25 | Acc:  11.35% | Loss: 2.3427 | Time:   79.7s | Comm: 0.675Gb
Round  30 | Acc:  21.31% | Loss: 2.4137 | Time:   98.1s | Comm: 0.810Gb
Round  35 | Acc:  10.06% | Loss: 3.5506 | Time:  114.2s | Comm: 0.944Gb
Round  40 | Acc:   9.74% | Loss: 2.9336 | Time:  127.5s | Comm: 1.079Gb
Round  45 | Acc:  24.69% | Loss: 2.2947 | Time:  142.8s | Comm: 1.214Gb
Round  50 | Acc:  14.32% | Loss: 2.1848 | Time:  157.0s | Comm: 1.349Gb
Round  55 | Acc:  10.87% | Loss: 2.2051 | Time:  172.5s | Comm:

KeyboardInterrupt: 

## 6. Experiment 2: Selection Strategies (Control Group)

Test selection strategies with a control group of 50 workers

In [ ]:
# Test selection strategies with control group (50 workers)
CONTROL_WORKERS = 50
selection_strategies = [
    RandomSelection(),
    CyclicSelection(),
    AdaptiveLAGSelection()
]

strategy_results = {}

print("\n" + "="*60)
print(f"EXPERIMENT 2: SELECTION STRATEGIES ({CONTROL_WORKERS} workers)")
print("="*60)

for strategy in selection_strategies:
    results = run_worker_selection_experiment(
        num_total_workers=CONTROL_WORKERS,
        selection_strategy=strategy,
        workers_per_round=None,  # 30% of total (15 workers)
        num_rounds=50,
        local_epochs=5,
        batch_size=32,
        learning_rate=0.003,
        target_accuracy=90.0,
        device=device
    )
    strategy_results[f'{CONTROL_WORKERS}_workers_{strategy.name}'] = results

print("\n" + "="*60)
print("✓ Selection strategy experiments completed!")
print("="*60)

## 7. Visualizations

In [ ]:
# Plot 1: Accuracy vs Time - Number of Workers
fig, ax = plt.subplots(figsize=(14, 8))

colors = plt.cm.viridis(np.linspace(0, 1, len(num_workers_list)))

for idx, num_workers in enumerate(num_workers_list):
    key = f'{num_workers}_workers_Random'
    if key in worker_results:
        results = worker_results[key]
        ax.plot(results['training_time_history'], results['accuracy_history'],
               marker='o', markersize=4, linewidth=2.5,
               label=f'{num_workers} Workers', color=colors[idx], alpha=0.8)

        # Mark convergence
        if results['convergence_round'] is not None:
            conv_idx = results['convergence_round'] - 1
            if conv_idx < len(results['training_time_history']):
                ax.scatter(results['training_time_history'][conv_idx],
                          results['accuracy_history'][conv_idx],
                          s=300, marker='*', color=colors[idx],
                          edgecolors='black', linewidths=2, zorder=5)

ax.axhline(y=90, color='red', linestyle='--', linewidth=2,
          label='Target (90%)', alpha=0.7)

ax.set_xlabel('Training Time (seconds)', fontsize=14, fontweight='bold')
ax.set_ylabel('Model Accuracy (%)', fontsize=14, fontweight='bold')
ax.set_title('Accuracy vs Training Time: Effect of Number of Workers',
            fontsize=16, fontweight='bold', pad=20)
ax.legend(loc='lower right', fontsize=11, framealpha=0.9)
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 100])

plt.tight_layout()
plt.savefig('accuracy_vs_time_num_workers.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: accuracy_vs_time_num_workers.png")

In [ ]:
# Plot 2: Accuracy vs Time - Selection Strategies (Control Group)
fig, ax = plt.subplots(figsize=(14, 8))

strategy_colors = {'Random': 'blue', 'Cyclic': 'green', 'Adaptive-LAG': 'red'}
strategy_markers = {'Random': 'o', 'Cyclic': 's', 'Adaptive-LAG': '^'}

for key, results in strategy_results.items():
    strategy_name = results['selection_strategy']
    ax.plot(results['training_time_history'], results['accuracy_history'],
           marker=strategy_markers[strategy_name], markersize=4, linewidth=2.5,
           label=f'{strategy_name} Selection',
           color=strategy_colors[strategy_name], alpha=0.8)

    # Mark convergence
    if results['convergence_round'] is not None:
        conv_idx = results['convergence_round'] - 1
        if conv_idx < len(results['training_time_history']):
            ax.scatter(results['training_time_history'][conv_idx],
                      results['accuracy_history'][conv_idx],
                      s=300, marker='*', color=strategy_colors[strategy_name],
                      edgecolors='black', linewidths=2, zorder=5)

ax.axhline(y=90, color='orange', linestyle='--', linewidth=2,
          label='Target (90%)', alpha=0.7)

ax.set_xlabel('Training Time (seconds)', fontsize=14, fontweight='bold')
ax.set_ylabel('Model Accuracy (%)', fontsize=14, fontweight='bold')
ax.set_title(f'Accuracy vs Training Time: Worker Selection Strategies ({CONTROL_WORKERS} Workers)',
            fontsize=16, fontweight='bold', pad=20)
ax.legend(loc='lower right', fontsize=11, framealpha=0.9)
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 100])

plt.tight_layout()
plt.savefig('accuracy_vs_time_selection_strategies.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: accuracy_vs_time_selection_strategies.png")

In [ ]:
# Plot 3: Communication Cost vs Number of Workers
fig, ax = plt.subplots(figsize=(14, 8))

worker_counts = []
comm_costs = []
final_accs = []

for num_workers in num_workers_list:
    key = f'{num_workers}_workers_Random'
    if key in worker_results:
        results = worker_results[key]
        worker_counts.append(num_workers)
        comm_costs.append(results['total_communication_cost'] / 1e9)
        final_accs.append(results['accuracy_history'][-1])

bars = ax.bar(range(len(worker_counts)), comm_costs,
              color=plt.cm.viridis(np.linspace(0, 1, len(worker_counts))),
              alpha=0.7, edgecolor='black', linewidth=2)

# Add accuracy labels
for i, (bar, acc) in enumerate(zip(bars, final_accs)):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
           f'{acc:.1f}%',
           ha='center', va='bottom', fontweight='bold')

ax.set_xlabel('Number of Workers', fontsize=14, fontweight='bold')
ax.set_ylabel('Total Communication Cost (Gigabits)', fontsize=14, fontweight='bold')
ax.set_title('Communication Cost vs Number of Workers',
            fontsize=16, fontweight='bold', pad=20)
ax.set_xticks(range(len(worker_counts)))
ax.set_xticklabels(worker_counts)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('communication_cost_vs_workers.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: communication_cost_vs_workers.png")

In [ ]:
# Plot 4: Worker Participation Distribution (for selection strategies)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (key, results) in enumerate(strategy_results.items()):
    strategy_name = results['selection_strategy']
    participation = list(results['worker_participation'].values())

    ax = axes[idx]
    ax.hist(participation, bins=20, color=strategy_colors[strategy_name],
           alpha=0.7, edgecolor='black')

    ax.axvline(results['participation_stats']['mean'],
              color='red', linestyle='--', linewidth=2, label='Mean')

    ax.set_xlabel('Number of Participations', fontsize=11, fontweight='bold')
    ax.set_ylabel('Number of Workers', fontsize=11, fontweight='bold')
    ax.set_title(f'{strategy_name}\n(μ={results["participation_stats"]["mean"]:.1f}, '
                f'σ={results["participation_stats"]["std"]:.1f})',
                fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle(f'Worker Participation Distribution by Selection Strategy ({CONTROL_WORKERS} Workers)',
            fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('worker_participation_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: worker_participation_distribution.png")

## 8. Summary Tables

In [ ]:
# Summary Table 1: Number of Workers
print("\n" + "="*90)
print("SUMMARY TABLE 1: Effect of Number of Workers (Random Selection)")
print("="*90)
print(f"{'Workers':<12} {'Selected/Round':<15} {'Conv. Round':<15} {'Time (s)':<12} {'Comm (Gb)':<12} {'Final Acc (%)':<15}")
print("-"*90)

for num_workers in num_workers_list:
    key = f'{num_workers}_workers_Random'
    if key in worker_results:
        results = worker_results[key]
        conv = results['convergence_round'] if results['convergence_round'] else 'N/A'
        selected = results['workers_per_round']
        time = f"{results['total_training_time']:.1f}"
        comm = f"{results['total_communication_cost']/1e9:.3f}"
        acc = f"{results['accuracy_history'][-1]:.2f}"

        print(f"{num_workers:<12} {selected:<15} {str(conv):<15} {time:<12} {comm:<12} {acc:<15}")

print("="*90)

# Summary Table 2: Selection Strategies
print("\n" + "="*90)
print(f"SUMMARY TABLE 2: Worker Selection Strategies ({CONTROL_WORKERS} Workers)")
print("="*90)
print(f"{'Strategy':<20} {'Conv. Round':<15} {'Time (s)':<12} {'Comm (Gb)':<12} {'Final Acc (%)':<15}")
print("-"*90)

for key, results in strategy_results.items():
    strategy = results['selection_strategy']
    conv = results['convergence_round'] if results['convergence_round'] else 'N/A'
    time = f"{results['total_training_time']:.1f}"
    comm = f"{results['total_communication_cost']/1e9:.3f}"
    acc = f"{results['accuracy_history'][-1]:.2f}"

    print(f"{strategy:<20} {str(conv):<15} {time:<12} {comm:<12} {acc:<15}")

print("="*90)

# Participation Fairness Analysis
print("\n" + "="*90)
print("WORKER PARTICIPATION FAIRNESS")
print("="*90)
print(f"{'Strategy':<20} {'Mean':<10} {'Std Dev':<10} {'Min':<10} {'Max':<10} {'Fairness Score':<15}")
print("-"*90)

for key, results in strategy_results.items():
    strategy = results['selection_strategy']
    stats = results['participation_stats']
    # Fairness score: lower std = more fair (normalized)
    fairness = 1 - (stats['std'] / stats['mean']) if stats['mean'] > 0 else 0

    print(f"{strategy:<20} {stats['mean']:<10.1f} {stats['std']:<10.2f} "
          f"{stats['min']:<10} {stats['max']:<10} {fairness:<15.3f}")

print("="*90)
print("Note: Fairness Score closer to 1.0 = more equal participation")
print("="*90)

## 9. Save Results

In [ ]:
# Combine all results
all_results = {**worker_results, **strategy_results}

# Save to JSON
with open('worker_selection_results.json', 'w') as f:
    json_results = {}
    for key, results in all_results.items():
        json_results[key] = {
            k: [float(v) if isinstance(v, (np.floating, np.integer)) else v
                for v in val] if isinstance(val, list) else
               (int(v) if isinstance(v, np.integer) else v) if not isinstance(val, dict) else val
            for k, val in results.items()
        }
    json.dump(json_results, f, indent=2)

print("✓ Results saved to 'worker_selection_results.json'")
print("✓ Plots saved:")
print("  - accuracy_vs_time_num_workers.png")
print("  - accuracy_vs_time_selection_strategies.png")
print("  - communication_cost_vs_workers.png")
print("  - worker_participation_distribution.png")
print("\nDownload files from the left sidebar!")